# 17. Взаимная информация: QMI и Z-MI

Две разные меры, partial trace без global density matrix, masks,
final-angle prefixes и passive observer.

In [1]:
import json, math
from pathlib import Path
import numpy as np
from QA_max_cut_lib import WeightedGraph
from QA_max_cut_lib.laboratory import QAOAWithMIObserverCPU
from QA_max_cut_lib.tools.mutual_information import *
ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent

## A. Product, Bell Φ/Ψ, CZ

QMI и Z-MI — разные величины; CZ plus имеет такие же Z probabilities,
как product plus, но другую QMI.

In [2]:
plus = np.ones(4) / 2
bell_phi = np.array([1, 0, 0, 1]) / math.sqrt(2)
bell_psi = np.array([0, 1, 1, 0]) / math.sqrt(2)
cz_plus = np.array([1, 1, 1, -1]) / 2
for name, psi in [("plus", plus), ("Φ", bell_phi), ("Ψ", bell_psi), ("CZ+", cz_plus)]:
    print(name, "QMI", quantum_mutual_information(psi, [0, 1], (0, 1)), "Z-MI", classical_from_probabilities(abs(psi) ** 2, [0, 1], (0, 1)))

plus QMI 0.0 Z-MI 0.0
Φ QMI 1.9999999999999998 Z-MI 1.0
Ψ QMI 1.9999999999999998 Z-MI 1.0
CZ+ QMI 1.9999999999999996 Z-MI 0.0


## B/C. Mixed pair RDM и entropy bug

In [3]:
ghz = np.zeros(8); ghz[0] = ghz[7] = 1 / math.sqrt(2)
print("GHZ pair QMI:", quantum_mutual_information(ghz, [0, 1, 2], (0, 1)))
state = np.zeros(8, dtype=complex); state[0] = state[5] = 1 / math.sqrt(2)
print("Bell(A,E) QMI A,B:", quantum_mutual_information(state, [0, 1, 2], (0, 1)))
print("S_B:", von_neumann_entropy(reduced_density_matrix(state, [1], [0, 1, 2])))

GHZ pair QMI: 1.0
Bell(A,E) QMI A,B: 2.220446049250313e-16
S_B: 3.2034265038149176e-16


## D. Complex RDM и labels

In [4]:
state = np.zeros(16, dtype=complex); state[0] = state[5] = 0.5; state[2] = state[7] = 0.5j
labels = [80, -7, 101, 10]
print(reduced_density_matrix(state, [1], labels))
print("pair QMI:", quantum_mutual_information(state, labels, (80, 101)))

[[0.5+0.j  0. -0.5j]
 [0. +0.5j 0.5+0.j ]]
pair QMI: 2.0


## E/F. Counts, masks, honest zero

In [5]:
print("counts MI:", classical_from_counts([[3, 1], [1, 3]]))
print("summary:", map_summary(np.array([2.0, 0.0, 0.0]), np.array([True, True, True])))

counts MI: 0.18872187554086717
summary: {'count': 3, 'mean': 0.6666666666666666, 'min': 0.0, 'max': 2.0, 'population_std': 0.9428090415820634, 'values': [2.0, 0.0, 0.0]}


## Passive QAOA observer

Baseline и observer имеют одни и те же selected angles, shots и solution;
добавляются только MI snapshots.

In [6]:
from QA_max_cut_lib.generators.graphs import GraphFactory
from QA_max_cut_lib.solvers.qaoa_cpu import QAOACPU
graph = GraphFactory.from_recipe(json.loads((ROOT / "configs/graphs/block_mean_field_6.json").read_text()))
config = json.loads((ROOT / "configs/solvers/qaoa_mi_cpu.json").read_text())
config["output"]["report"] = 0
plain_config = dict(config); plain_config["type"] = "qaoa_cpu"; plain_config.pop("diagnostics", None)
plain = QAOACPU(plain_config); plain_solution = plain.solve(graph)
observer = QAOAWithMIObserverCPU(config); observed_solution = observer.solve(graph)
print("same solution:", observed_solution == plain_solution)
print("shots:", observer.result["resources"]["shots_used"], plain.result["resources"]["shots_used"])
print("frames:", observer.result["mi_summary"]["snapshot_count"])
print("first frame summary:", observer.result["mi_summary"]["quantum"][0])

same solution: True
shots: 352 352
frames: 3
first frame summary: {'count': 15, 'mean': 0.0, 'min': 0.0, 'max': 0.0, 'population_std': 0.0, 'values': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}


## Ограничение

QMI — exact simulator diagnostic; Z-MI из finite shots является
эмпирической оценкой и не заменяет QMI. Старые frames RQAOA относятся
к своему graph version и не переносятся на новый compact index.